In [1]:
%run "/home/nichettg/Ubuntu/Laboratorio/moduli.py"

# 1. Lettura Dati

In [2]:
def nome (stringa):

    stringa_nuova = stringa.replace(f'./Data/','').replace('.csv','')
    return stringa_nuova

In [3]:
input_files = [f"./Data/{nome_file}" for nome_file in sorted(os.listdir(f'./Data'))]
data = {}

for input_file in input_files:
    letti = pd.read_csv(input_file)
    data[nome(input_file)] = {}


    for key in letti.keys():
        colonna = letti[key]
        try:
            # prova a convertirla in float
            data[nome(input_file)][key] = colonna.astype(float).to_numpy()
        except ValueError:
            # se non ci riesce, tieni le stringhe
            data[nome(input_file)][key] = colonna.astype(str).to_list()

#### Media

# 2. Analisi Dati

In [4]:
err = 0

In [ ]:
file_output = "analizzati.txt"
f.inizializza_output(file_output)
def gaussiana(B,x):
    return B[0] * np.exp(-(x - B[1])**2 / (2 * B[2]**2))
for (titolo,dati) in data.items():
    fig,ax = plt.subplots(1, 1,  figsize=(10, 5))
    xdata = [dati["pixel"],
            np.full_like(dati["pixel"],1/np.sqrt(12))]
    ydata = [dati["intensita"],
            np.sqrt( err*2 + (0.05*dati["intensita"])**2 )]
    gf.parametri_grafico(ax,titolo,xlabel="Pixel",ylabel="Intensità (u.a.)")
    ax.errorbar(xdata[0], ydata[0], xerr=xdata[1], yerr=ydata[1], color="red", fmt=".", **gf.default_error_params())

    x = xdata[0]
    y = ydata[0]
    A0 = np.max(y)
    mu0 = x[np.argmax(y)]
    sigma0 = (np.max(x) - np.min(x)) / 6
    beta0 = [A0, mu0, sigma0]

    anal = f.fit(xdata,ydata,gaussiana,beta0, chi=False)

    with open(file_output,'a') as out:
        out.write(f"Fit {titolo}\n")
        out.write(f"\tx del vertice = {gf.formatta_errore(x[0],x[1],txt=True)}\n")
        out.write(f"\ty del vertice = {gf.formatta_errore(y[0],y[1],txt=True)}\n")
        out.write(f"\ta = {gf.formatta_errore(a,sa,txt=True)}\n")
        out.write(f"\tb = {gf.formatta_errore(b,sb,txt=True)}\n")
        out.write(f"\tc = {gf.formatta_errore(c,sc,txt=True)}\n\n")

    x = np.linspace(np.min(xdata[0]), np.max(xdata[0]), 500)
    y = gaussiana([anal[0][0], anal[1][0], anal[2][0]], x)
    ax.plot(x, y, color="blue")

    del xdata,ydata,x,y,a,b,c